# Stage 0 lean suite — overnight runner

Reduced suite designed to fit in a single overnight Colab session. **16 training runs total**:

**Phase A — `lean_compare` (8 runs at 300k):** the 3-method comparison plus the gru-vs-none ablation for CAC, on `armed_corridor` and `phase_crossing`. Matches the cross-branch consensus in `main`/`angelic`/`Ade`'s `run_short.sh`.

**Phase D — `lean_confirm` (8 runs at 1M):** same 4 configs × 2 benchmarks, run to long-horizon for headline numbers.

## Configuration matrix (per benchmark)

| #   | Method               | training_mode | temporal_encoding |
| --- | -------------------- | ------------- | ----------------- |
| 1   | no_concept           | (n/a)         | gru               |
| 2   | vanilla_freeze       | two_phase     | gru               |
| 3   | concept_actor_critic | two_phase     | gru               |
| 4   | concept_actor_critic | two_phase     | none              |

## Time budget on observed throughput

- L4: ~10 h total (Phase A ~2.5 h, Phase D ~7.5 h)
- A100: ~3-4 h total

## Front-end compatibility

Works under native Colab (browser), VS Code + Colab extension, and local Jupyter (CPU smoke only).

## Resume on disconnect

All cells are idempotent. If a session disconnects, reconnect, re-run cells 1-3 (per-session ritual), then re-run whichever phase cell was active. Completed runs are skipped via `eval.json` presence check on Drive.


In [ ]:
from pathlib import Path

p = Path("/content/repo/colab/run_suite.py")
lines = p.read_text().splitlines()

for i, line in enumerate(lines):
    if "stream_train_logs={\\'on\\'" in line:
        lines[i] = '        f"stream_train_logs={\'on\' if args.stream_train_logs else \'off\'}"'

p.write_text("\n".join(lines) + "\n")

!python -m py_compile /content/repo/colab/run_suite.py


In [23]:
# 1) Mount Drive + define suite-wide variables (re-run after every reconnect)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/concept_critic'
except ImportError:
    print('Not running on Colab — using local /tmp for outputs')
    DRIVE_ROOT = '/tmp/concept_critic'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Suite-wide variables. Used by every round cell. Edit here if you need to override.
OUTPUT_DIR  = f'{DRIVE_ROOT}/stage0'
BENCHMARKS  = 'armed_corridor phase_crossing'
MAX_MINUTES = 660   # ~11h, safely under Colab Pro session cap

# Export Python variables so shell commands in later cells can use $OUTPUT_DIR,
# $BENCHMARKS, and $MAX_MINUTES reliably.
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
os.environ['OUTPUT_DIR'] = OUTPUT_DIR
os.environ['BENCHMARKS'] = BENCHMARKS
os.environ['MAX_MINUTES'] = str(MAX_MINUTES)

print('output root:', DRIVE_ROOT)
print('OUTPUT_DIR :', OUTPUT_DIR)
print('BENCHMARKS :', BENCHMARKS)
print('MAX_MINUTES:', MAX_MINUTES)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
output root: /content/drive/MyDrive/concept_critic
OUTPUT_DIR : /content/drive/MyDrive/concept_critic/stage0
BENCHMARKS : armed_corridor phase_crossing
MAX_MINUTES: 660


In [24]:
# 2) Locate or fetch the repo. Handles three cases:
#    - Native Colab: clone into /content/repo if not already present
#    - VS Code + Colab extension: workspace already synced, train.py is in cwd or an ancestor
#    - Local Jupyter (no Colab): same as above — train.py is somewhere up the tree from the notebook
import os, subprocess, sys

REPO_URL = 'https://github.com/AdeX11/concept_critic_models.git'
BRANCH   = 'domingo-experimental'   # pin a commit SHA for reproducibility, e.g. 'b70d8ac'

def _find_repo_root() -> str | None:
    cur = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(cur, 'train.py')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    if os.path.exists('/content/repo/train.py'):
        return '/content/repo'
    return None

REPO_DIR = _find_repo_root()
if REPO_DIR is None:
    if not os.path.isdir('/content'):
        raise RuntimeError(
            'train.py not found in cwd ancestors and /content does not exist. '
            'Open this notebook from inside the cloned repo, or run it on a Colab runtime.'
        )
    REPO_DIR = '/content/repo'
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print('repo  :', REPO_DIR)
print('head  :', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())
print('branch:', subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip())

# Require live streaming support so the suite cell shows training progress.
run_suite_path = os.path.join(REPO_DIR, 'colab', 'run_suite.py')
with open(run_suite_path) as f:
    run_suite_src = f.read()
if '--stream_train_logs' not in run_suite_src:
    raise RuntimeError(
        'This runtime has an old colab/run_suite.py without --stream_train_logs. '
        'Sync the updated repo/notebook into Colab, then re-run cells 1 and 2.'
    )
RUN_SUITE_LIVE_FLAG = '--stream_train_logs'
os.environ['RUN_SUITE_LIVE_FLAG'] = RUN_SUITE_LIVE_FLAG
subprocess.check_call([sys.executable, '-m', 'py_compile', run_suite_path])
print('live train logs: enabled')

repo  : /content/repo
head  : 3e7cf744981720cbfe29dd75f98c2123a1224098
branch: domingo-experimental


CalledProcessError: Command '['/usr/bin/python3', '-m', 'py_compile', '/content/repo/colab/run_suite.py']' returned non-zero exit status 1.

In [22]:
# 3) Install deps + smoke + measure throughput on the GPU you actually got
!bash colab/setup.sh

=== python ===
Python 3.12.13
=== installing requirements ===
ERROR: Operation cancelled by user


## Overnight unattended — Phase A then Phase D

**Recommended runtime:** A100 if available (~3-4 h total). L4 works but ~10 h total.

This single cell runs both phases sequentially. If a Colab session disconnects mid-way, reconnect, redo cells 1-3, then re-run this cell — already-completed runs are skipped automatically.

Each phase honors `MAX_MINUTES` independently to stay under session caps.


In [ ]:
# Phase A: lean_compare (8 runs at 300k)
!python colab/run_suite.py \
    --round lean_compare \
    --benchmarks $BENCHMARKS \
    --output_dir $OUTPUT_DIR \
    --max_minutes $MAX_MINUTES \
    $RUN_SUITE_LIVE_FLAG

# Phase D: lean_confirm (8 runs at 1M)
!python colab/run_suite.py \
    --round lean_confirm \
    --benchmarks $BENCHMARKS \
    --output_dir $OUTPUT_DIR \
    --max_minutes $MAX_MINUTES \
    $RUN_SUITE_LIVE_FLAG

## Final aggregation

Run after both phases are complete. Produces a CSV with all 16 runs and prints per-method/per-benchmark counts.


In [ ]:
!python cluster/aggregate.py $OUTPUT_DIR --csv-out $OUTPUT_DIR/lean_full.csv
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/lean_full.csv')
print('total runs:', len(df))
print()
print(df.groupby(['method', 'temporal_encoding', 'benchmark_id']).size().rename('count'))
print()
print(df[['method','temporal_encoding','benchmark_id','mean_reward','success_rate','dominant_action_fraction']].to_string(index=False))

## Granular control (optional)

If you want to run the phases separately — e.g. inspect Phase A results before launching Phase D, or restart only Phase D — use these cells in place of the overnight cell above.

### Phase A only


In [ ]:
!python colab/run_suite.py \
    --round lean_compare \
    --benchmarks $BENCHMARKS \
    --output_dir $OUTPUT_DIR \
    --max_minutes $MAX_MINUTES \
    $RUN_SUITE_LIVE_FLAG

### Inspect Phase A results before Phase D


In [ ]:
!python cluster/aggregate.py $OUTPUT_DIR --csv-out $OUTPUT_DIR/lean_phaseA.csv
!head -20 $OUTPUT_DIR/lean_phaseA.csv

### Phase D only (1M long-horizon — switch to A100 first if available)


In [ ]:
!python colab/run_suite.py \
    --round lean_confirm \
    --benchmarks $BENCHMARKS \
    --output_dir $OUTPUT_DIR \
    --max_minutes $MAX_MINUTES \
    $RUN_SUITE_LIVE_FLAG

## Monitoring & troubleshooting

**Live training progress:** the phase cells must print `stream_train_logs=on`, then stream each active `train.py` run directly into the notebook output and save the same lines to that run's `train.log`. Watch the `[iter ...]` lines for timesteps, rolling reward, episodes, fps, policy/value loss, concept actor/critic loss, entropy loss, and concept MSE.

**Suite progress between runs** — open a new cell and run:

```python
!tail -10 $OUTPUT_DIR/_suite_progress.jsonl
```

**Tail a specific run log** if you reconnect or want the latest persisted training lines. Use Python brace substitution for `RUN`; plain `$RUN` is a shell environment variable and may expand to empty:

```python
RUN = 'no_concept_two_phase_gru_phase_crossing_seed42'
!tail -80 "{OUTPUT_DIR}/{RUN}/train.log"
```

**Check TensorBoard event files for a run:**

```python
RUN = 'no_concept_two_phase_gru_phase_crossing_seed42'
!find "{OUTPUT_DIR}/{RUN}/tb" -maxdepth 1 -type f -name 'events.out.tfevents*' -ls
```

**Text summary of TensorBoard scalars:**

```python
RUN = 'no_concept_two_phase_gru_phase_crossing_seed42'
!python colab/summarize_tensorboard.py "{OUTPUT_DIR}/{RUN}"
```

**TensorBoard for any run:**

```python
RUN = 'no_concept_two_phase_gru_phase_crossing_seed42'
%load_ext tensorboard
%tensorboard --logdir "{OUTPUT_DIR}/{RUN}/tb"
```

**Inspecting a failed run** (look for `FAIL(rc=…)` in suite output):

```python
RUN = '<run_dir_name>'
!tail -40 "{OUTPUT_DIR}/{RUN}/train.log"
```

**Disconnect recovery:** reconnect → re-run cells 1, 2, 3 → re-run the phase cell that was active. Completed runs (those with `eval.json` on Drive) are skipped via the resume-skip check; any in-flight run is restarted from scratch (no per-run checkpoint resume yet — see `STAGE_0_PLAN.md` for that backlog item).
